In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from joblib import dump

In [2]:
data = pd.read_csv("Data/processed_data")

X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "DTI", "home_ownership", "purpose"]] 
Y = data["is_loss"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42, stratify=Y)
Y_test = pd.DataFrame(Y_test)
X_train.head()

,annual_income_ru,loan_ammount_ru,int_rate_ru,DTI,home_ownership,purpose
10024,799200.0,41440.0,0.206695,0.051852,OWN,Debt consolidation
7031,858400.0,148000.0,0.198811,0.172414,MORTGAGE,Debt consolidation
31908,651200.0,279720.0,0.145346,0.429545,MORTGAGE,other
14271,426240.0,44400.0,0.159250,0.104167,OWN,Debt consolidation
14948,695600.0,74000.0,0.143196,0.106383,RENT,Debt consolidation


In [ ]:
#Стандартизация для чисел:
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

#One-Hot кодирование:
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

#Объединяем Стандартизацию и One-Hot в один обработчик
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, [0,1,2,3]),
        ('cat', categorical_transformer, [4,5])
    ]
)

#Создаем итоговый Pipeline с моделью
pipeline = TunedThresholdClassifierCV(estimator=Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        random_state=42,
        class_weight="balanced"
    ))
]),scoring="f1", n_jobs=-1, cv=Stra)
pipeline.fit(X_train, Y_train)
prediction = pipeline.predict(X_test)
recall = recall_score(Y_test,prediction)
precision = precision_score(Y_test,prediction)
f1_score = f1_score(Y_test, prediction)
print(recall)
print(precision)
print(f1_score)

0.6114028507126782
0.2168706758914316
0.32017285405617757


In [5]:
model = pipeline.named_steps['classifier']  # или 'regressor'def get_feature_weights_basic(pipeline):
def get_feature_weights_basic(pipeline):
    # Получаем модель и препроцессор
    model = pipeline.named_steps['classifier']
    preprocessor = pipeline.named_steps['preprocessor']
    
    coefficients = model.coef_[0]  # для бинарной классификации
    
    # Получаем имена признаков после трансформации
    feature_names = []
    
    for name, transformer, columns in preprocessor.transformers_:
        if hasattr(transformer, 'named_steps'):
            # Если transformer - это Pipeline (например, с imputer + encoder)
            if 'onehot' in transformer.named_steps:
                encoder = transformer.named_steps['onehot']
            else:
                # Если нет onehot, но есть encoder
                encoder = transformer.named_steps.get('encoder', None)
        else:
            # Если transformer непосредственно OneHotEncoder
            encoder = transformer
        
        if name == 'num':
            # Просто добавляем имена числовых признаков
            feature_names.extend(columns)
        elif name == 'cat' and hasattr(encoder, 'get_feature_names_out'):
            try:
                # Способ 1: Получаем имена из encoder.feature_names_in_
                if hasattr(encoder, 'feature_names_in_'):
                    # OneHotEncoder запомнил имена признаков при обучении
                    encoded_features = encoder.get_feature_names_out(encoder.feature_names_in_)
                else:
                    # Способ 2: Используем сохраненные columns (если они совпадают)
                    encoded_features = encoder.get_feature_names_out(columns)
            except ValueError:
                # Способ 3: Если ничего не работает, используем общий метод
                encoded_features = encoder.get_feature_names_out()
            
            feature_names.extend(encoded_features)
    
    # Создаем DataFrame
    weights_df = pd.DataFrame({
        'feature': feature_names,
        'weight': coefficients
    })
    
    return weights_df
weights = get_feature_weights_basic(pipeline)
weights.sort_values(ascending=False, by = "weight")

,feature,weight
13,purpose_small business,0.617404
2,2,0.586514
14,purpose_vacation,0.273633
4,home_ownership_OTHER,0.255340
1,1,0.022407
10,purpose_house,0.020539
12,purpose_other,0.000445
3,home_ownership_MORTGAGE,-0.069302
5,home_ownership_OWN,-0.077119
6,home_ownership_RENT,-0.089633


In [31]:
features = pd.DataFrame(data={
        "annual_income_ru":[2580000],
        "loan_ammount_ru":[100000],
        "int_rate_ru":[0.2],
        "home_ownership":["OWN"],
        "purpose":["car"]
})
print(pipeline.predict_proba(features))
data["purpose"].value_counts()

[[0.58952245 0.41047755]]


purpose
Debt consolidation    18213
other                  9261
credit card            4998
major purchase         2110
small business         1776
car                    1497
house                   366
vacation                352
Name: count, dtype: int64

In [ ]:
data["home_ownership"].value_counts()

home_ownership
RENT        18439
MORTGAGE    17198
OWN          2838
OTHER          98
Name: count, dtype: int64

In [36]:
path = "models/LogisticRegression.joblib"
dump(pipeline, path)

['models/LogisticRegression.joblib']